In [ ]:
import os
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# load data
def load_data(df : str)-> pd.DataFrame:
    df = pd.read_csv(df)
    return df

# Train Test Function
def split_data(X, y, test_size=0.2, random_state=42):   #random_state=42 provides same split every time and reproducable result
    return train_test_split(
        X, y, test_size=test_size,random_state=random_state,stratify=y #Split the data in such a way that the class distribution of y stays the same in both train and test sets.
    )

# Model ROC AUC and Accuracy Function
def validate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train) # fit the model on the training data
    y_pred = model.predict(X_test) # predict the labels for the test set
    acc = accuracy_score(y_test, y_pred)
    try:
        auc = roc_auc_score(y_test, model.predict_proba(X_test), multi_class='ovr') # get the probability of the positive class
    except:
        auc = "ROC AUC score cannot be calculated for this model."

    return auc, acc

# Impute missing data function add 5% missing values to the dataset and impute them using SimpleImputer
def impute_data(X_train, X_test):
    rng = np.random.default_rng(42)
    X_gaps_train = X_train.copy()
    mask = rng.random(X_gaps_train.shape) < 0.05
    X_gaps_train[mask] = np.nan
    X_gaps_test = X_train.copy()
    mask = rng.random(X_gaps_test.shape) < 0.05
    X_gaps_test[mask] = np.nan
    print(f"Missing before imputation on train datasets: {X_gaps_train.isnull().sum().sum()}")
    print(f"Missing before imputation on test datasets: {X_gaps_test.isnull().sum().sum()}")
    imputer=SimpleImputer(strategy='mean')
    X_Imputed_train = imputer.fit_transform(X_gaps_train)
    X_Imputed_test = imputer.transform(X_gaps_test)
    print(f"Missing after imputation on train datasets: {pd.DataFrame(X_Imputed_train).isnull().sum().sum()}")
    print(f"Missing after imputation on test datasets: {pd.DataFrame(X_Imputed_test).isnull().sum().sum()}")
    return X_Imputed_train, X_Imputed_test

# main function 
def main():
    input_dataset = '../data/raw/iris_dataset.csv'
    df = load_data(input_dataset)
    X = df.drop(columns=["target", "target_name"])
    y = df["target"]
    # Model A: raw features (no scaling)
    X_train, X_test, y_train, y_test = split_data(X, y)
    print(f"Dataset : {X.shape[0]} rows, {X.shape[1]} features")
    print(f"Train   : {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
    model = LogisticRegression(max_iter=200)
    roc_auc, acc = validate_model(model, X_train, y_train, X_test, y_test)
    print(f"ROC AUC Score RAW features: {roc_auc}")
    print(f"Accuracy RAW features: {acc}")
    # Model B: scaled features
    scaler=StandardScaler()
    scaler.fit(X_train) # fit the scaler on the training data
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test) # transform the test data using the same scaler
    model_scaled = LogisticRegression(max_iter=200)
    roc_auc_scaled, acc_scaled = validate_model(model_scaled, X_train_scaled, y_train, X_test_scaled, y_test)
    print(f"ROC AUC Score Scaled features: {roc_auc_scaled}")
    print(f"Accuracy Scaled features: {acc_scaled}")
    # Model C: imputed features
    #impute_data(X_train , X_test)
    X_train_imputed, X_test_imputed = impute_data(X_train, X_test)
    # DecisionTreeClassifier does not require feature scaling, so we can use the imputed data directly without scaling.
    model_tree = DecisionTreeClassifier(random_state=42)
    roc_auc_tree, acc_tree = validate_model(model_tree, X_train, y_train, X_test, y_test)
    print(f"ROC AUC Score Decision Tree: {roc_auc_tree}")
    print(f"Accuracy Decision Tree: {acc_tree}")
    # Random Forest Classifier
    model_rf = RandomForestClassifier(random_state=42)
    roc_auc_rf, acc_rf = validate_model(model_rf, X_train, y_train, X_test, y_test)
    print(f"ROC AUC Score Random Forest: {roc_auc_rf}")
    print(f"Accuracy Random Forest: {acc_rf}")
    # SVN and KNN do not support missing values, so we cannot use the imputed data for these models. We will only compare the results of the Decision Tree and Random Forest models with the raw and scaled features.
    # Compare the results of the three models
    # Summary of results
    print("\nSummary of Results:")
    print(f"Model A (Raw features) - ROC AUC: {roc_auc}, Accuracy: {acc}")
    print(f"Model B (Scaled features) - ROC AUC: {roc_auc_scaled}, Accuracy: {acc_scaled}")
    print(f"Model C (Imputed features) - Decision Tree ROC AUC: {roc_auc_tree}, Accuracy: {acc_tree}")
    print(f"Model C (Imputed features) - Random Forest ROC AUC: {roc_auc_rf}, Accuracy: {acc_rf}")

if __name__ == "__main__":
    main()









Dataset : 150 rows, 4 features
Train   : 120 rows | Test: 30 rows
ROC AUC Score RAW features: 1.0
Accuracy RAW features: 0.9666666666666667
ROC AUC Score Scaled features: 0.9966666666666667
Accuracy Scaled features: 0.9333333333333333
Missing before imputation on train datasets: 21
Missing before imputation on test datasets: 30
Missing after imputation on train datasets: 0
Missing after imputation on test datasets: 0
ROC AUC Score Decision Tree: 0.9499999999999998
Accuracy Decision Tree: 0.9333333333333333
ROC AUC Score Random Forest: 0.9866666666666667
Accuracy Random Forest: 0.9

Summary of Results:
Model A (Raw features) - ROC AUC: 1.0, Accuracy: 0.9666666666666667
Model B (Scaled features) - ROC AUC: 0.9966666666666667, Accuracy: 0.9333333333333333
Model C (Imputed features) - Decision Tree ROC AUC: 0.9499999999999998, Accuracy: 0.9333333333333333
Model C (Imputed features) - Random Forest ROC AUC: 0.9866666666666667, Accuracy: 0.9


: 